# **Prédiction de la Durée d'Hospitalisation**
## Comparaison de Modèles de Machine Learning

**Objectif** : Estimer la durée d'hospitalisation (`lengthofstay`) d'un patient en fonction de ses antécédents médicaux et mesures biologiques.

**Approche pédagogique** : Tester plusieurs modèles pour identifier le plus performant.

---
## 1. Imports et Configuration

In [ ]:
# Bibliothèques de base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modèles
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# XGBoost (installer si nécessaire : !pip install xgboost)
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost non installé. Exécuter: !pip install xgboost")
    XGBOOST_AVAILABLE = False

# Métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuration d'affichage
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Imports réussis")

: 

---
## 2. Chargement et Exploration des Données

In [ ]:
# Charger le dataset
# Adapter le chemin selon votre environnement
# df = pd.read_csv('/content/drive/MyDrive/cmv-ia/LengthOfStay.csv')  # Google Colab
df = pd.read_csv('LengthOfStay.csv')  # Local

print(f"Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.info()

In [ ]:
df.head()

In [ ]:
# Statistiques descriptives
df.describe()

In [ ]:
# Vérifier les valeurs manquantes
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✅")

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(df['lengthofstay'], bins=17, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Durée d\'hospitalisation (jours)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution de lengthofstay')
axes[0].axvline(df['lengthofstay'].mean(), color='red', linestyle='--', label=f'Moyenne: {df["lengthofstay"].mean():.2f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['lengthofstay'], vert=True)
axes[1].set_ylabel('Durée d\'hospitalisation (jours)')
axes[1].set_title('Boxplot de lengthofstay')

plt.tight_layout()
plt.show()

print(f"\nStatistiques de la variable cible :")
print(f"  Moyenne : {df['lengthofstay'].mean():.2f} jours")
print(f"  Médiane : {df['lengthofstay'].median():.2f} jours")
print(f"  Min-Max : {df['lengthofstay'].min()} - {df['lengthofstay'].max()} jours")

---
## 3. Préparation des Données (Feature Engineering)

In [ ]:
# Copie du dataframe pour le preprocessing
df_processed = df.copy()

# Colonnes à supprimer (identifiants, dates)
cols_to_drop = ['eid', 'vdate', 'discharged']
df_processed = df_processed.drop(columns=cols_to_drop)

print(f"Colonnes supprimées : {cols_to_drop}")
print(f"Colonnes restantes : {df_processed.shape[1]}")

In [ ]:
# Identifier les colonnes catégorielles
cat_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
print(f"Colonnes catégorielles : {cat_cols}")

for col in cat_cols:
    print(f"\n{col} - Valeurs uniques : {df_processed[col].nunique()}")
    print(df_processed[col].value_counts())

In [ ]:
# Encodage des variables catégorielles

# gender : F/M -> 0/1
df_processed['gender'] = df_processed['gender'].map({'F': 0, 'M': 1})

# rcount : 0, 1, 2, 3, 4, 5+ -> conversion numérique
df_processed['rcount'] = df_processed['rcount'].replace('5+', '5').astype(int)

# facid : One-Hot Encoding pour l'établissement
df_processed = pd.get_dummies(df_processed, columns=['facid'], prefix='facid', drop_first=True)

print("✅ Encodage terminé")
print(f"Shape après encodage : {df_processed.shape}")
df_processed.head()

In [ ]:
# Vérifier les types de données finaux
print("Types de données :")
print(df_processed.dtypes.value_counts())

---
## 4. Analyse des Corrélations

In [ ]:
# Matrice de corrélation avec la variable cible
correlations = df_processed.corr()['lengthofstay'].sort_values(ascending=False)
print("Corrélations avec lengthofstay :\n")
print(correlations)

In [ ]:
# Visualisation des corrélations (top features)
plt.figure(figsize=(10, 8))
top_corr = correlations.drop('lengthofstay').head(15)
colors = ['green' if x > 0 else 'red' for x in top_corr.values]
plt.barh(top_corr.index, top_corr.values, color=colors, alpha=0.7)
plt.xlabel('Corrélation avec lengthofstay')
plt.title('Top 15 Features les plus corrélées avec la durée d\'hospitalisation')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap des corrélations (subset pour lisibilité)
plt.figure(figsize=(14, 12))
correlation_matrix = df_processed.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

---
## 5. Préparation des Données pour la Modélisation

In [ ]:
# Séparation Features / Target
X = df_processed.drop('lengthofstay', axis=1)
y = df_processed['lengthofstay']

print(f"Features (X) : {X.shape}")
print(f"Target (y)   : {y.shape}")
print(f"\nFeatures utilisées ({len(X.columns)}) :")
print(list(X.columns))

In [ ]:
# Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape[0]:,} échantillons")
print(f"Test  : {X_test.shape[0]:,} échantillons")

In [ ]:
# Normalisation des features (important pour certains modèles)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Normalisation effectuée (StandardScaler)")

---
## 6. Définition des Fonctions d'Évaluation

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Évalue un modèle et retourne les métriques.
    """
    # Prédictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Métriques Train
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Métriques Test
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    
    return {
        'Model': model_name,
        'Train_MAE': train_mae,
        'Train_RMSE': train_rmse,
        'Train_R2': train_r2,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_R2': test_r2
    }

def print_model_results(results):
    """
    Affiche les résultats d'un modèle.
    """
    print(f"\n{'='*60}")
    print(f"📊 {results['Model']}")
    print(f"{'='*60}")
    print(f"\n  TRAIN:")
    print(f"    MAE  : {results['Train_MAE']:.4f} jours")
    print(f"    RMSE : {results['Train_RMSE']:.4f} jours")
    print(f"    R²   : {results['Train_R2']:.4f}")
    print(f"\n  TEST:")
    print(f"    MAE  : {results['Test_MAE']:.4f} jours")
    print(f"    RMSE : {results['Test_RMSE']:.4f} jours")
    print(f"    R²   : {results['Test_R2']:.4f}")
    
    # Indicateur d'overfitting
    overfit_gap = results['Train_R2'] - results['Test_R2']
    if overfit_gap > 0.1:
        print(f"\n  ⚠️  Overfitting potentiel (gap R²: {overfit_gap:.4f})")
    else:
        print(f"\n  ✅ Bonne généralisation (gap R²: {overfit_gap:.4f})")

# Stockage des résultats
all_results = []

---
## 7. Entraînement et Comparaison des Modèles

### 7.1 Régression Linéaire (Baseline)

In [ ]:
# 1. RÉGRESSION LINÉAIRE
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

lr_results = evaluate_model(lr_model, X_train_scaled, X_test_scaled, y_train, y_test, "Régression Linéaire")
all_results.append(lr_results)
print_model_results(lr_results)

In [ ]:
# Coefficients de la régression linéaire
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 coefficients les plus importants :")
print(coef_df.head(10).to_string(index=False))

### 7.2 Ridge Regression (Régularisation L2)

In [ ]:
# 2. RIDGE REGRESSION
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

ridge_results = evaluate_model(ridge_model, X_train_scaled, X_test_scaled, y_train, y_test, "Ridge Regression")
all_results.append(ridge_results)
print_model_results(ridge_results)

### 7.3 Lasso Regression (Régularisation L1)

In [ ]:
# 3. LASSO REGRESSION
lasso_model = Lasso(alpha=0.01, random_state=42, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

lasso_results = evaluate_model(lasso_model, X_train_scaled, X_test_scaled, y_train, y_test, "Lasso Regression")
all_results.append(lasso_results)
print_model_results(lasso_results)

# Features éliminées par Lasso
n_zero = sum(lasso_model.coef_ == 0)
print(f"\n📌 Lasso a éliminé {n_zero} features (coefficients = 0)")

### 7.4 ElasticNet (L1 + L2)

In [ ]:
# 4. ELASTICNET
elastic_model = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=10000)
elastic_model.fit(X_train_scaled, y_train)

elastic_results = evaluate_model(elastic_model, X_train_scaled, X_test_scaled, y_train, y_test, "ElasticNet")
all_results.append(elastic_results)
print_model_results(elastic_results)

### 7.5 K-Nearest Neighbors (KNN)

In [ ]:
# 5. KNN REGRESSOR
knn_model = KNeighborsRegressor(n_neighbors=10, weights='distance')
knn_model.fit(X_train_scaled, y_train)

knn_results = evaluate_model(knn_model, X_train_scaled, X_test_scaled, y_train, y_test, "KNN Regressor")
all_results.append(knn_results)
print_model_results(knn_results)

### 7.6 Decision Tree

In [ ]:
# 6. DECISION TREE
dt_model = DecisionTreeRegressor(max_depth=10, min_samples_split=20, random_state=42)
dt_model.fit(X_train, y_train)  # Pas besoin de scaling pour les arbres

dt_results = evaluate_model(dt_model, X_train, X_test, y_train, y_test, "Decision Tree")
all_results.append(dt_results)
print_model_results(dt_results)

In [ ]:
# Feature importance - Decision Tree
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(dt_importance['Feature'][:15], dt_importance['Importance'][:15], color='forestgreen', alpha=0.7)
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances - Decision Tree')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 7.7 Random Forest

In [ ]:
# 7. RANDOM FOREST
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_results = evaluate_model(rf_model, X_train, X_test, y_train, y_test, "Random Forest")
all_results.append(rf_results)
print_model_results(rf_results)

In [ ]:
# Feature importance - Random Forest
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(rf_importance['Feature'][:15], rf_importance['Importance'][:15], color='steelblue', alpha=0.7)
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 7.8 Gradient Boosting

In [ ]:
# 8. GRADIENT BOOSTING
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_train, y_train)

gb_results = evaluate_model(gb_model, X_train, X_test, y_train, y_test, "Gradient Boosting")
all_results.append(gb_results)
print_model_results(gb_results)

### 7.9 XGBoost

In [ ]:
# 9. XGBOOST
xgb_model = None
if XGBOOST_AVAILABLE:
    xgb_model = XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    
    xgb_results = evaluate_model(xgb_model, X_train, X_test, y_train, y_test, "XGBoost")
    all_results.append(xgb_results)
    print_model_results(xgb_results)
else:
    print("⚠️ XGBoost non disponible - installer avec: !pip install xgboost")

### 7.10 Support Vector Regression (SVR)

⚠️ **Note** : SVR est lent sur de grands datasets. On utilise un échantillon.

In [ ]:
# 10. SVR (sur échantillon car lent)
# Sous-échantillonnage pour SVR
sample_size = 10000
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)

X_train_sample = X_train_scaled[sample_idx]
y_train_sample = y_train.iloc[sample_idx]

svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_model.fit(X_train_sample, y_train_sample)

# Évaluation sur le test complet
svr_results = evaluate_model(svr_model, X_train_sample, X_test_scaled, y_train_sample, y_test, "SVR (échantillon)")
all_results.append(svr_results)
print_model_results(svr_results)
print(f"\n📌 SVR entraîné sur {sample_size:,} échantillons (au lieu de {len(X_train):,})")

---
## 8. Comparaison Finale des Modèles

In [ ]:
# Tableau récapitulatif
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Test_R2', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("📊 TABLEAU COMPARATIF DES MODÈLES (trié par R² Test)")
print("="*80)

# Formater pour l'affichage
display_df = results_df.copy()
for col in ['Train_MAE', 'Test_MAE', 'Train_RMSE', 'Test_RMSE']:
    display_df[col] = display_df[col].round(4)
for col in ['Train_R2', 'Test_R2']:
    display_df[col] = display_df[col].round(4)

print(display_df.to_string(index=False))

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = results_df['Model']
colors = plt.cm.viridis(np.linspace(0, 0.8, len(models)))

# R² Score
ax1 = axes[0, 0]
x = np.arange(len(models))
width = 0.35
ax1.bar(x - width/2, results_df['Train_R2'], width, label='Train', alpha=0.8)
ax1.bar(x + width/2, results_df['Test_R2'], width, label='Test', alpha=0.8)
ax1.set_ylabel('R² Score')
ax1.set_title('R² Score par Modèle')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, 1)

# MAE
ax2 = axes[0, 1]
ax2.bar(x - width/2, results_df['Train_MAE'], width, label='Train', alpha=0.8)
ax2.bar(x + width/2, results_df['Test_MAE'], width, label='Test', alpha=0.8)
ax2.set_ylabel('MAE (jours)')
ax2.set_title('Mean Absolute Error par Modèle')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=45, ha='right')
ax2.legend()

# RMSE
ax3 = axes[1, 0]
ax3.bar(x - width/2, results_df['Train_RMSE'], width, label='Train', alpha=0.8)
ax3.bar(x + width/2, results_df['Test_RMSE'], width, label='Test', alpha=0.8)
ax3.set_ylabel('RMSE (jours)')
ax3.set_title('Root Mean Squared Error par Modèle')
ax3.set_xticks(x)
ax3.set_xticklabels(models, rotation=45, ha='right')
ax3.legend()

# Écart Train-Test (Overfitting indicator)
ax4 = axes[1, 1]
overfit_gap = results_df['Train_R2'] - results_df['Test_R2']
colors_overfit = ['red' if g > 0.1 else 'green' for g in overfit_gap]
ax4.bar(models, overfit_gap, color=colors_overfit, alpha=0.7)
ax4.axhline(y=0.1, color='red', linestyle='--', label='Seuil overfitting')
ax4.set_ylabel('Gap R² (Train - Test)')
ax4.set_title('Indicateur d\'Overfitting')
ax4.set_xticklabels(models, rotation=45, ha='right')
ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Meilleur modèle
best_model = results_df.iloc[0]
print("\n" + "="*60)
print("🏆 MEILLEUR MODÈLE")
print("="*60)
print(f"\n  Modèle : {best_model['Model']}")
print(f"  R² Test : {best_model['Test_R2']:.4f}")
print(f"  MAE Test : {best_model['Test_MAE']:.4f} jours")
print(f"  RMSE Test : {best_model['Test_RMSE']:.4f} jours")

---
## 9. Analyse des Prédictions du Meilleur Modèle

In [ ]:
# Utilisons le modèle Gradient Boosting ou Random Forest pour l'analyse
best_trained_model = gb_model  # ou rf_model
y_pred = best_trained_model.predict(X_test)

# Scatter plot : Valeurs réelles vs Prédictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, y_pred, alpha=0.3, s=5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Prédiction parfaite')
axes[0].set_xlabel('Valeurs Réelles (jours)')
axes[0].set_ylabel('Prédictions (jours)')
axes[0].set_title('Valeurs Réelles vs Prédictions')
axes[0].legend()

# Distribution des résidus
residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('Résidus (jours)')
axes[1].set_ylabel('Fréquence')
axes[1].set_title(f'Distribution des Résidus\nMoyenne: {residuals.mean():.3f}, Std: {residuals.std():.3f}')

plt.tight_layout()
plt.show()

---
## 10. Conclusions et Recommandations

### Résumé des observations :

1. **Régression Linéaire** : Bonne baseline, interprétable, mais performances limitées sur ce dataset non-linéaire.

2. **Ridge/Lasso/ElasticNet** : Apportent peu d'amélioration car il y a peu de multicolinéarité problématique.

3. **KNN** : Sensible au bruit et à la dimensionnalité.

4. **Decision Tree** : Risque d'overfitting si non contrôlé.

5. **Random Forest / Gradient Boosting / XGBoost** : Meilleurs performances, capturent les relations non-linéaires.

### Recommandations :

- **Pour l'interprétabilité** : Régression Linéaire ou Decision Tree (peu profond)
- **Pour la performance** : Gradient Boosting ou XGBoost avec tuning des hyperparamètres
- **Features les plus importantes** : `rcount` (nombre de réadmissions), mesures biologiques, diagnostics secondaires

In [ ]:
# Export des résultats
results_df.to_csv('model_comparison_results.csv', index=False)
print("✅ Résultats exportés dans 'model_comparison_results.csv'")

---
## 11. 💾 Sauvegarde du Modèle XGBoost pour l'API

In [ ]:
import os

os.makedirs("models", exist_ok=True)

# Sauvegarde au format JSON natif XGBoost
xgb_model.save_model("models/xgboost_los_model.json")

file_size = os.path.getsize("models/xgboost_los_model.json") / 1024
print(f"✅ Modèle sauvegardé : models/xgboost_los_model.json")
print(f"   Taille            : {file_size:.1f} Ko")
print(f"   Features utilisées: {list(X.columns)}")


In [ ]:
from xgboost import XGBRegressor

# Rechargement depuis le fichier JSON
model_loaded = XGBRegressor()
model_loaded.load_model("models/xgboost_los_model.json")

# Test sur 5 patients du jeu de test
sample = X_test.iloc[:5]
predictions = model_loaded.predict(sample)
real_values  = y_test.iloc[:5].values

print("Vérification sur 5 patients :")
print(f"{'Patient':<10} {'Prédit':>12} {'Réel':>8} {'Écart':>10}")
print("-" * 44)
for i, (pred, real) in enumerate(zip(predictions, real_values)):
    print(f"{i+1:<10} {pred:>10.2f}j {real:>6.0f}j {abs(pred-real):>8.2f}j")

print(f"\n✅ Modèle rechargé et fonctionnel — prêt pour l'API !")


### 🚀 Utilisation dans l'API

In [ ]:
# ──────────────────────────────────────────────────────
# Snippet à copier dans ton API (FastAPI / Flask)
# ──────────────────────────────────────────────────────

snippet = '''
from xgboost import XGBRegressor
import pandas as pd

# Charger une seule fois au démarrage de l'API
model = XGBRegressor()
model.load_model("models/xgboost_los_model.json")

# Colonnes attendues (dans le même ordre qu'à l'entraînement)
FEATURE_COLUMNS = ''' + str(list(X.columns)) + '''

def predict_length_of_stay(patient_data: dict) -> float:
    df_input = pd.DataFrame([patient_data])[FEATURE_COLUMNS]
    prediction = model.predict(df_input)
    return round(float(prediction[0]), 2)
'''

print(snippet)
